In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

#Sprint 1

In [ ]:
!pip -q install pandas pyarrow scikit-learn
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [ ]:
# Ajusta o caminho conforme a tua pasta
parquet_path = "/content/drive/MyDrive/Planapp-main/data_sample/cision_news_rows.parquet"
sample = pd.read_parquet(parquet_path, engine="pyarrow")

In [ ]:
sample.head(-100)

,id,titulo,noticia,link,data_publicacao,tipo_meio,autores,valor_publicitario,guid,categoria
0,117766326,Novo regime de arrendamento com opção de compr...,O novo regime de arrendamento com opção de com...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1105.40,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
1,117766609,TRIBUNAL DETECTA OBRAS IRREGULARES EM SANTANA,Tribunal detecta obras irregulares em Santana ...,No link available,2025-06-19 00:00:00+00:00,Imprensa,Miguel Fernandes Luís,1323.10,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
2,117766528,Relatório final da Comissão Parlamentar de Inq...,“Os processos de contratualização subjacente à...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1097.60,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
3,117766532,Valença prevê concluir em 2026 obras no Centro...,A reabilitação e ampliação do Centro de Saúde ...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,100.00,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
4,117766576,CHUMBADA MOÇÃO DE REJEIÇÃO DO PCP,Após a votação à moção de rejeição do PCP ao P...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1445.20,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
...,...,...,...,...,...,...,...,...,...,...
19895,119165457,PS apresentou Rui Miguel Cruz para a presidênc...,O PS apresentou Rui Miguel Cruz como candidato...,https://centralpress.pt/page/107604/redacao/20...,2025-09-16 00:00:00+00:00,Internet,Samuel Guiomar,111.82,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
19896,119165646,Carta aberta une personalidades portuguesas à ...,"Dezenas de portugueses, numa carta aberta, anu...",https://bomdia.eu/carta-aberta-une-personalida...,2025-09-16 00:00:00+00:00,Internet,Fabiana Bravo,600.00,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
19897,119165554,Ventura aponta a Gouveia e Melo com risco e se...,"Tal como parecia inevitável há muito, e foi av...",https://www.dn.pt/pol%C3%ADtica/ventura-aponta...,2025-09-16 00:00:00+00:00,Internet,Leonardo Ralha,16937.38,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
19898,119165563,Governo disponibiliza 30 milhões de euros para...,O Governo vai disponibilizar 30 milhões de eur...,https://odigital.sapo.pt/governo-disponibiliza...,2025-09-16 00:00:00+00:00,Internet,sem autor,272.44,http://pt.cision.com/cp2013/clippingdetails.as...,NaN


In [ ]:
print(sample.shape)
print(sample.columns)
display(sample.head())
display(sample.dtypes)
print(sample.isna().sum().sort_values(ascending=False).head(10))

# Exemplo: ver duplicados
print("Duplicados por corpo de noticia:", sample.duplicated(subset=["noticia"]).sum())

(20000, 10)
Index(['id', 'titulo', 'noticia', 'link', 'data_publicacao', 'tipo_meio',
       'autores', 'valor_publicitario', 'guid', 'categoria'],
      dtype='object')


,id,titulo,noticia,link,data_publicacao,tipo_meio,autores,valor_publicitario,guid,categoria
0,117766326,Novo regime de arrendamento com opção de compr...,O novo regime de arrendamento com opção de com...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1105.4,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
1,117766609,TRIBUNAL DETECTA OBRAS IRREGULARES EM SANTANA,Tribunal detecta obras irregulares em Santana ...,No link available,2025-06-19 00:00:00+00:00,Imprensa,Miguel Fernandes Luís,1323.1,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
2,117766528,Relatório final da Comissão Parlamentar de Inq...,“Os processos de contratualização subjacente à...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1097.6,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
3,117766532,Valença prevê concluir em 2026 obras no Centro...,A reabilitação e ampliação do Centro de Saúde ...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,100.0,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
4,117766576,CHUMBADA MOÇÃO DE REJEIÇÃO DO PCP,Após a votação à moção de rejeição do PCP ao P...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1445.2,http://pt.cision.com/cp2013/clippingdetails.as...,NaN


,0
id,int64
titulo,object
noticia,object
link,object
data_publicacao,"datetime64[ns, UTC]"
tipo_meio,object
autores,object
valor_publicitario,float64
guid,object
categoria,float64


categoria             20000
id                        0
titulo                    0
noticia                   0
data_publicacao           0
link                      0
tipo_meio                 0
autores                   0
valor_publicitario        0
guid                      0
dtype: int64
Duplicados por corpo de noticia: 201


In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
import re

# Escolhe a coluna base
col_texto = 'titulo'

# Normalização simples do texto: minúsculas, remover múltiplos espaços, remover pontuação leve
def normalizar(s):
    s = str(s).lower()
    s = re.sub(r"[\:\;\,\.\!\?\(\)\[\]\-\–\—\"\']", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

sample[col_texto] = sample[col_texto].astype(str).fillna("").map(normalizar)

from nltk.corpus import stopwords
stopwords_pt = set(stopwords.words('portuguese'))

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

#Vetorização TF-IDF e cálculo da matriz de similaridade
vectorizer = TfidfVectorizer(stop_words=list(stopwords_pt), ngram_range=(1,2), max_features=20000, min_df=2)
X = vectorizer.fit_transform(sample[col_texto])

# Similaridade e novo limiar (mais baixo aumenta grupos)
S = cosine_similarity(X, dense_output=False)
threshold = 0.72

n = S.shape[0]
visitados = np.zeros(n, dtype=bool)
grupos = []
for i in range(n):
    if visitados[i]:
        continue
    similares = S[i].toarray().ravel() >= threshold
    grupo_idxs = np.where(similares)[0].tolist()
    visitados[grupo_idxs] = True
    grupos.append(grupo_idxs)


In [ ]:
# para contagens
from collections import Counter
sizes = Counter(len(g) for g in grupos)
print('Distribuição de tamanhos de grupo:', sizes)
print('Grupos totais:', len(grupos))
print('Grupos com 1 notícia:', sum(1 for g in grupos if len(g)==1))


Distribuição de tamanhos de grupo: Counter({1: 10839, 2: 1924, 3: 511, 4: 192, 5: 114, 6: 74, 8: 41, 0: 38, 7: 36, 9: 26, 10: 25, 11: 17, 16: 11, 12: 10, 14: 10, 18: 9, 15: 8, 17: 7, 13: 5, 22: 4, 21: 4, 24: 3, 31: 2, 36: 2, 32: 2, 28: 2, 49: 1, 41: 1, 27: 1, 26: 1, 30: 1, 44: 1, 20: 1, 19: 1, 25: 1})
Grupos totais: 13925
Grupos com 1 notícia: 10839


In [ ]:
#criar o grupo id no dataframe
# Mapa notícia -> id do grupo (a partir da lista `grupos`)
grupo_id_map = {idx: gid for gid, g in enumerate(grupos) for idx in g}
sample["grupo_id"] = sample.index.map(grupo_id_map.get).astype("Int64")


In [ ]:
# Tabela com número de notícias por grupo
contagem_por_grupo = (
    sample.groupby("grupo_id")
          .size()
          .reset_index(name="num_noticias")
)

# Lista com os ids dos grupos que têm exatamente 32 notícias p.e.
grupos_32 = contagem_por_grupo.loc[
    contagem_por_grupo["num_noticias"] == 32, "grupo_id"
].tolist()

print("Grupos com x notícias:", grupos_32)
print("Total de grupos com x notícias:", len(grupos_32))


Grupos com x notícias: [3365, 7556]
Total de grupos com x notícias: 2


In [ ]:
# Garante que cada notícia tem o id do grupo
# Passo 1: Gerar o campo grupo_id para cada notícia
grupo_id_map = {idx: gid for gid, g in enumerate(grupos) for idx in g}
sample["grupo_id"] = sample.index.map(grupo_id_map.get).astype("Int64")  # aceita NA

sample["is_lider"] = False  # Inicializa a coluna is_lider

In [ ]:

gid = grupos_32[0] #0 é a primeria posicao, 1 a segunda e etc
exemplo_4 = sample.loc[sample["grupo_id"] == gid, ["grupo_id","is_lider","titulo","data_publicacao"]].head(50)
display(exemplo_4)



,grupo_id,is_lider,titulo,data_publicacao
4424,3365,False,urgências encerradas,2025-07-06 18:09:21+00:00
4430,3365,False,urgências encerradas,2025-07-06 19:59:28+00:00
4433,3365,False,urgências encerradas,2025-07-06 20:04:58+00:00
6158,3365,False,urgências encerradas,2025-07-12 12:58:16+00:00
6166,3365,False,urgências encerradas,2025-07-12 13:24:11+00:00
6175,3365,False,urgências encerradas,2025-07-12 15:02:33+00:00
6223,3365,False,urgências encerradas,2025-07-12 20:13:32+00:00
6224,3365,False,urgências encerradas,2025-07-12 20:14:47+00:00
8338,3365,False,urgências encerradas,2025-07-20 10:59:33+00:00
9508,3365,False,urgências encerradas,2025-07-25 20:02:56+00:00


In [ ]:
#escolher líder por grupo (primeira publicada)
def lider_primeira_publicada_seguro(df, grupo, col_data='data_publicacao'):
    import pandas as pd
    if not isinstance(grupo, (list, tuple, np.ndarray)) or len(grupo) == 0:
        return None
    idx_validos = [int(i) for i in grupo if 0 <= int(i) < len(df)]
    if len(idx_validos) == 0:
        return None
    datas = pd.to_datetime(df.loc[idx_validos, col_data], errors='coerce')
    valid_mask = datas.notna()
    if not valid_mask.any():
        return idx_validos[0]
    min_idx = datas[valid_mask].idxmin()
    return int(min_idx)

col_data = "data_publicacao"
lideres = [lider_primeira_publicada_seguro(sample, g, col_data=col_data) for g in grupos]

sample["is_lider"] = False  # inicializa
lideres_validos = [i for i in lideres if i is not None and 0 <= int(i) < len(sample)]
sample.loc[lideres_validos, "is_lider"] = True

# Finalmente: vê as colunas relevantes
print(sample[["grupo_id", "is_lider", col_texto, col_data]].head(30))


    grupo_id  is_lider                                             titulo  \
0          0      True  novo regime de arrendamento com opção de compr...   
1          1      True      tribunal detecta obras irregulares em santana   
2          2      True  relatório final da comissão parlamentar de inq...   
3          3      True  valença prevê concluir em 2026 obras no centro...   
4          4      True                  chumbada moção de rejeição do pcp   
5          5      True  investimento superior a 2 milhões de euros pom...   
6          6      True  34 º aniversário da elevação de pombal a cidad...   
7          7      True  comissão decide instaurar ação contra portugal...   
8          8      True  investigadores desenvolvem base de dados e mod...   
9          9      True  visita às obras requalificação do parque de fr...   
10        10      True                     governo um programa reformista   
11        11      True          faleceu o psiquiatra e escritor pio abreu   

In [ ]:
detalhes = sample[sample["grupo_id"] == 7556].head(50)
print(detalhes[["titulo", "data_publicacao"]])
#todas as noticias de cada grupo - neste caso o grupo (id) 7556 tem 32 noticias como vimos

                                  titulo           data_publicacao
10563  análise aos incêndios em portugal 2025-07-30 18:12:08+00:00
10593  análise aos incêndios em portugal 2025-07-30 21:06:57+00:00
11550  análise aos incêndios em portugal 2025-08-04 19:10:30+00:00
11567  análise aos incêndios em portugal 2025-08-04 22:33:06+00:00
12045  análise aos incêndios em portugal 2025-08-07 23:01:29+00:00
12906  análise aos incêndios em portugal 2025-08-11 19:05:22+00:00
13093  análise aos incêndios em portugal 2025-08-13 19:28:41+00:00
13102  análise aos incêndios em portugal 2025-08-13 21:23:02+00:00
13103  análise aos incêndios em portugal 2025-08-13 21:42:37+00:00
13107  análise aos incêndios em portugal 2025-08-13 22:34:52+00:00
13319  análise aos incêndios em portugal 2025-08-14 23:17:56+00:00
13892  análise aos incêndios em portugal 2025-08-16 19:11:11+00:00
14012  análise aos incêndios em portugal 2025-08-17 17:11:01+00:00
14021  análise aos incêndios em portugal 2025-08-17 20:20:14+0

In [ ]:
sample.head(40)

,id,titulo,noticia,link,data_publicacao,tipo_meio,autores,valor_publicitario,guid,categoria,grupo_id,is_lider
0,117766326,novo regime de arrendamento com opção de compr...,O novo regime de arrendamento com opção de com...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1105.4,http://pt.cision.com/cp2013/clippingdetails.as...,NaN,0,True
1,117766609,tribunal detecta obras irregulares em santana,Tribunal detecta obras irregulares em Santana ...,No link available,2025-06-19 00:00:00+00:00,Imprensa,Miguel Fernandes Luís,1323.1,http://pt.cision.com/cp2013/clippingdetails.as...,NaN,1,True
2,117766528,relatório final da comissão parlamentar de inq...,“Os processos de contratualização subjacente à...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1097.6,http://pt.cision.com/cp2013/clippingdetails.as...,NaN,2,True
3,117766532,valença prevê concluir em 2026 obras no centro...,A reabilitação e ampliação do Centro de Saúde ...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,100.0,http://pt.cision.com/cp2013/clippingdetails.as...,NaN,3,True
4,117766576,chumbada moção de rejeição do pcp,Após a votação à moção de rejeição do PCP ao P...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1445.2,http://pt.cision.com/cp2013/clippingdetails.as...,NaN,4,True
5,117750707,investimento superior a 2 milhões de euros pom...,OInstituto Politécnico de Leiria lançou o conc...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,100.0,http://pt.cision.com/cp2013/clippingdetails.as...,NaN,5,True
6,117750710,34 º aniversário da elevação de pombal a cidad...,Perceber como era Pombal há 34 anos e ver a ci...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,314.0,http://pt.cision.com/cp2013/clippingdetails.as...,NaN,6,True
7,117768156,comissão decide instaurar ação contra portugal...,CURIOSIDADES\n\nComissão decide instaurar ação...,https://terrasdohomem.pt/2025/06/19/comissao-d...,2025-06-19 00:00:00+00:00,Internet,sem autor,102.0,http://pt.cision.com/cp2013/clippingdetails.as...,NaN,7,True
8,117768279,investigadores desenvolvem base de dados e mod...,Uma equipa de investigadores da Faculdade de C...,https://guimaraesagora.pt/investigadores-desen...,2025-06-19 00:00:00+00:00,Internet,sem autor,114.0,http://pt.cision.com/cp2013/clippingdetails.as...,NaN,8,True
9,117749968,visita às obras requalificação do parque de fr...,"O presidente da Câmara Municipal, Paulo Ferrei...",No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,100.0,http://pt.cision.com/cp2013/clippingdetails.as...,NaN,9,True


_____________________

_____________________